In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
import librosa

processor = WhisperProcessor.from_pretrained("openai/whisper-base")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")

# Загрузка аудио
audio, sr = librosa.load("cv-invalid/sample-000000.mp3", sr=16000)
input_features = processor(audio, sampling_rate=16000, return_tensors="pt").input_features

# Генерация транскрипции
with torch.no_grad():
    predicted_ids = model.generate(input_features)
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print(transcription)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

 you


In [ ]:
import torch
import torchaudio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import warnings
# (опционально) подавляем конкретные предупреждения о логгитс-процессорах
warnings.filterwarnings("ignore", message="A custom logits processor of type*")

# Загрузка модели и процессора
model_name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# Перемещаем модель на GPU, если доступно
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Загрузка аудио (пример с torchaudio)
audio_path = "cv-invalid/sample-000002.mp3"
waveform, sample_rate = torchaudio.load(audio_path)

# Ресемплинг до 16 кГц (требование Whisper)
if sample_rate != 16000:
    resampler = torchaudio.transforms.Resample(sample_rate, 16000)
    waveform = resampler(waveform)

# Берём первый канал, если стерео
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)

# Извлекаем признаки
inputs = processor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
input_features = inputs.input_features.to(device)
# attention_mask не нужен для одного файла, но можно передать, если есть

# Генерация транскрипции с явным указанием языка и задачи
with torch.no_grad():
    generated_ids = model.generate(
        input_features,
        language="en",          # укажите нужный язык
        task="transcribe",       # или "translate"
        # Другие параметры: max_length, num_beams и т.д.
    )

# Декодируем в текст
transcription = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Транскрибация:", transcription)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Транскрибация:  Then suddenly he noticed it with his touch.


In [1]:
# ==========================================================
# 0. IMPORTS
# ==========================================================
import os
import pandas as pd
import numpy as np
import torch
import librosa
import evaluate

from datasets import Dataset, DatasetDict
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    Trainer,
    TrainingArguments
)

# ==========================================================
# 1. CONFIG
# ==========================================================
dataset_path = r"D:/tms/tms_study/voice to text/common-voice"
csv_file     = "cv-invalid.csv"     # ИМЕННО CSV
clips_dir    = "clips"

model_name   = "facebook/wav2vec2-base-960h"
output_dir  = "./wav2vec2-common-voice"

num_samples = 1000   # 0 = все данные

# ==========================================================
# 2. LOAD CSV
# ==========================================================
csv_path = os.path.join(dataset_path, csv_file)
df = pd.read_csv(csv_path)

# обязательные колонки Common Voice CSV
assert "filename" in df.columns
assert "text" in df.columns

df = df[["filename", "text"]]

if num_samples > 0:
    df = df.head(num_samples)

# путь к аудио
def build_audio_path(fname):
    return os.path.join(dataset_path, fname)

df["audio_path"] = df["filename"].apply(build_audio_path)

# фильтр существующих файлов
exists_mask = df["audio_path"].apply(os.path.exists)
print(f"❌ Не найдено аудио: {(~exists_mask).sum()}")
df = df[exists_mask].reset_index(drop=True)
print(f"✅ Используется {len(df)} примеров")

assert len(df) > 0, "Нет валидных аудиофайлов"

# ==========================================================
# 3. DATASET
# ==========================================================
dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(test_size=0.15, seed=42)
dataset = DatasetDict(
    train=dataset["train"],
    eval=dataset["test"],
)

# ==========================================================
# 4. PROCESSOR + MODEL
# ==========================================================
processor = Wav2Vec2Processor.from_pretrained(model_name)

model = Wav2Vec2ForCTC.from_pretrained(
    model_name,
    pad_token_id=processor.tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
)

# ==========================================================
# 5. PREPARE BATCH (CORRECT FOR CTC)
# ==========================================================
def prepare_batch(batch):
    # -----------------------------
    # 1️⃣ Загрузка аудио
    # -----------------------------
    audio, sr = librosa.load(batch["audio_path"], sr=16000)

    # -----------------------------
    # 2️⃣ Токенизация аудио
    # -----------------------------
    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors=None,
        return_attention_mask=False
    )

    # -----------------------------
    # 3️⃣ Токенизация текста (labels)
    # -----------------------------
    labels = processor.tokenizer(
        batch["text"],
        add_special_tokens=False
    ).input_ids

    # -----------------------------
    # 4️⃣ Возврат словаря
    # -----------------------------
    return {
        "input_values": inputs.input_values[0],
        "labels": labels
    }

dataset = dataset.map(
    prepare_batch,
    remove_columns=["filename", "text", "audio_path"],
    desc="Preparing dataset",
)

# ==========================================================
# 6. DATA COLLATOR (CTC)
# ==========================================================
def data_collator(features):
    input_values = [f["input_values"] for f in features]
    labels = [f["labels"] for f in features]

    # ✅ Padding для аудио
    batch = processor.pad(
        {"input_values": input_values},
        padding=True,
        return_tensors="pt"
    )

    # ✅ Padding для labels и замена pad_token_id на -100
    max_len = max(len(l) for l in labels)
    labels_tensor = torch.full((len(labels), max_len), -100)
    for i, l in enumerate(labels):
        labels_tensor[i, :len(l)] = torch.tensor(l, dtype=torch.long)

    batch["labels"] = labels_tensor
    return batch

# ==========================================================
# 7. METRICS
# ==========================================================
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    logits = pred.predictions
    pred_ids = np.argmax(logits, axis=-1)

    pred_str = processor.batch_decode(pred_ids)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {
        "wer": wer,
        "cer": cer,
    }

# ==========================================================
# 8. TRAINING ARGS
# ==========================================================
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    eval_steps=50,          # evaluate every 50 steps
    save_steps=50,          # save every 50 steps
    learning_rate=3e-5,
    num_train_epochs=10,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=False,  # старый transformers
    report_to="none",
)

# ==========================================================
# 9. TRAINER
# ==========================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    tokenizer=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ==========================================================
# 10. TRAIN + METRICS
# ==========================================================
if __name__ == "__main__":
    print("🚀 Start training")
    trainer.train()

    # Сохранение модели
    trainer.save_model(output_dir)
    processor.save_pretrained(output_dir)

    # Оценка на eval
    metrics = trainer.evaluate()
    print("✅ Training finished")
    print("Метрики на eval:")
    print(metrics)

c:\ProgramData\miniconda3\envs\whisper-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


❌ Не найдено аудио: 0
✅ Используется 1000 примеров


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Preparing dataset: 100%|██████████| 150/150 [00:01<00:00, 114.90 examples/s]
C:\Temp\ipykernel_804\1814709727.py:192: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Start training


Step,Training Loss
25,17.227400
50,4.170700
75,2.485800
100,3.423700
125,2.388800
150,2.877800
175,2.012600
200,3.197700
225,2.968800
250,2.974000


✅ Training finished
Метрики на eval:
{'eval_loss': 1.975725531578064, 'eval_wer': 1.0, 'eval_cer': 1.0, 'eval_runtime': 19.6734, 'eval_samples_per_second': 7.625, 'eval_steps_per_second': 7.625, 'epoch': 10.0}


In [2]:
# ==========================================================
# 11. ПРОВЕРКА PREDICTIONS НА EVAL
# ==========================================================
def show_eval_predictions(trainer, processor, dataset_eval, num_examples=5):
    """
    Выводит несколько примеров из eval: предсказание, правильный текст, WER и CER.
    """
    import random
    import numpy as np
    from datasets import Dataset
    
    # Случайные примеры
    indices = random.sample(range(len(dataset_eval)), min(num_examples, len(dataset_eval)))
    
    for idx in indices:
        sample = dataset_eval[idx]
        input_values = torch.tensor(sample["input_values"]).unsqueeze(0)  # batch=1
        input_values = input_values.to(trainer.model.device)
        
        # Предсказание
        with torch.no_grad():
            logits = trainer.model(input_values).logits
        pred_ids = torch.argmax(logits, dim=-1)
        pred_str = processor.batch_decode(pred_ids)[0]
        
        # Правильный текст
        label_ids = sample["labels"]
        # -100 заменяем на pad_token_id для декодирования
        label_ids = [id if id != -100 else processor.tokenizer.pad_token_id for id in label_ids]
        label_str = processor.batch_decode([label_ids], group_tokens=False)[0]
        
        # WER/CER для одного примера
        wer_metric = evaluate.load("wer")
        cer_metric = evaluate.load("cer")
        wer = wer_metric.compute(predictions=[pred_str], references=[label_str])
        cer = cer_metric.compute(predictions=[pred_str], references=[label_str])
        
        print("--------------------------------------------------")
        print(f"Пример #{idx}")
        print(f"Правильный текст: {label_str}")
        print(f"Предсказанный текст: {pred_str}")
        print(f"WER: {wer:.3f}, CER: {cer:.3f}")
        print("--------------------------------------------------\n")

# Запуск функции на eval
show_eval_predictions(trainer, processor, dataset["eval"], num_examples=5)

--------------------------------------------------
Пример #28
Правильный текст: <unk><unk><unk><unk><unk> <unk><unk><unk><unk><unk>'<unk> <unk><unk><unk><unk><unk><unk> <unk><unk><unk><unk><unk><unk><unk><unk> <unk><unk><unk> <unk><unk><unk> <unk><unk><unk><unk><unk><unk><unk>
Предсказанный текст: 
WER: 1.000, CER: 1.000
--------------------------------------------------

--------------------------------------------------
Пример #6
Правильный текст: <unk><unk><unk> <unk><unk><unk><unk><unk> <unk><unk><unk><unk><unk><unk> <unk><unk><unk> <unk><unk><unk><unk><unk><unk> <unk><unk><unk><unk><unk><unk><unk><unk> <unk><unk><unk><unk><unk><unk> <unk><unk> <unk><unk><unk><unk>
Предсказанный текст: 
WER: 1.000, CER: 1.000
--------------------------------------------------

--------------------------------------------------
Пример #70
Правильный текст: <unk><unk> <unk><unk><unk><unk> <unk><unk><unk><unk> <unk><unk> <unk><unk><unk> <unk><unk><unk><unk><unk> <unk><unk><unk><unk><unk> <unk><unk> <u

In [2]:
# ==========================================================
# 0. IMPORTS
# ==========================================================
import os
import torch
import torchaudio
import pandas as pd
import numpy as np
import soundfile as sf
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    Trainer,
    TrainingArguments,
)
# ==========================================================
# 1. PATHS & CONFIG
# ==========================================================
BASE_DIR = r"D:/tms/tms_study/voice to text/common-voice"
TRAIN_CSV = "cv-other-train.csv"
EVAL_CSV = "cv-valid-dev.csv"
TRAIN_AUDIO_DIR = os.path.join(BASE_DIR, "cv-other-train")
EVAL_AUDIO_DIR = os.path.join(BASE_DIR, "cv-valid-dev")
VAD_OUT_DIR = os.path.join(BASE_DIR, "wav2vec2-common-voice-vad")
os.makedirs(VAD_OUT_DIR, exist_ok=True)
SEG_TRAIN_DIR = os.path.join(VAD_OUT_DIR, "train")
SEG_EVAL_DIR = os.path.join(VAD_OUT_DIR, "eval")
os.makedirs(SEG_TRAIN_DIR, exist_ok=True)
os.makedirs(SEG_EVAL_DIR, exist_ok=True)
MODEL_NAME = "facebook/wav2vec2-base-960h"
SAMPLE_RATE = 16000
MAX_SEGMENT_SEC = 15.0
NUM_TRAIN_SAMPLES = 200
NUM_EVAL_SAMPLES = 100
# ==========================================================
# 2. LOAD SILERO VAD
# ==========================================================
vad_model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)
(get_speech_timestamps,
 save_audio,
 read_audio,
 VADIterator,
 collect_chunks) = utils
# ==========================================================
# 3. LOAD CSV
# ==========================================================
def load_csv(csv_name, audio_dir):
    # Пропускаем строки с ошибками
    df = pd.read_csv(os.path.join(BASE_DIR, csv_name), on_bad_lines="skip")
   
    # Переименовываем колонки в ожидаемые
    df = df.rename(columns={"filename": "path", "text": "sentence"})
   
    # Берем только нужные колонки
    df = df[["path", "sentence"]].dropna()
   
    # Полные пути к аудио
    df["audio_path"] = df["path"].apply(lambda x: os.path.join(audio_dir, x))
   
    # Оставляем только существующие файлы
    df = df[df["audio_path"].apply(os.path.exists)]
   
    return df.reset_index(drop=True)
train_df = load_csv(TRAIN_CSV, TRAIN_AUDIO_DIR)
eval_df = load_csv(EVAL_CSV, EVAL_AUDIO_DIR)
# Adjust samples if eval is empty
if len(eval_df) == 0:
    total_needed = NUM_TRAIN_SAMPLES + NUM_EVAL_SAMPLES
    train_df = train_df.head(total_needed)
    eval_df = train_df.tail(NUM_EVAL_SAMPLES).copy()
    train_df = train_df.head(NUM_TRAIN_SAMPLES)
else:
    train_df = train_df.head(NUM_TRAIN_SAMPLES)
    eval_df = eval_df.head(NUM_EVAL_SAMPLES)
print(f"Train samples: {len(train_df)}")
print(f"Eval samples : {len(eval_df)}")
# ==========================================================
# 4. VAD SEGMENTATION
# ==========================================================
def vad_segment(df, out_dir):
    rows = []
    for idx, row in df.iterrows():
        wav = read_audio(row["audio_path"], sampling_rate=SAMPLE_RATE)
        timestamps = get_speech_timestamps(
            wav,
            vad_model,
            sampling_rate=SAMPLE_RATE
        )
        if not timestamps:
            continue
        speech_chunks = []
        for ts in timestamps:
            dur = (ts["end"] - ts["start"]) / SAMPLE_RATE
            if dur > MAX_SEGMENT_SEC:
                continue
            speech_chunks.append(wav[ts["start"]:ts["end"]])
        if not speech_chunks:
            continue
        concat_speech = torch.cat(speech_chunks) if len(speech_chunks) > 1 else speech_chunks[0]
        total_dur = len(concat_speech) / SAMPLE_RATE
        if total_dur > MAX_SEGMENT_SEC:
            continue
        out_name = f"{idx}.wav"
        out_path = os.path.join(out_dir, out_name)
        sf.write(out_path, concat_speech.numpy(), SAMPLE_RATE)
        rows.append({
            "audio_path": out_path,
            "text": row["sentence"]
        })
    return pd.DataFrame(rows)
print("🔪 VAD train")
train_seg = vad_segment(train_df, SEG_TRAIN_DIR)
print("🔪 VAD eval")
eval_seg = vad_segment(eval_df, SEG_EVAL_DIR)
# ==========================================================
# 5. DATASET
# ==========================================================
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_seg),
    "eval": Dataset.from_pandas(eval_seg),
})
# ==========================================================
# 6. PROCESSOR + MODEL
# ==========================================================
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    pad_token_id=processor.tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
)
# ==========================================================
# 7. PREPARE BATCH
# ==========================================================
def prepare_batch(batch):
    speech, sr = torchaudio.load(batch["audio_path"])
    speech = speech.squeeze()
    if sr != SAMPLE_RATE:
        speech = torchaudio.functional.resample(speech, sr, SAMPLE_RATE)
    speech = speech.numpy()
    inputs = processor(
        speech,
        sampling_rate=SAMPLE_RATE
    )
    labels = processor(text=batch["text"]).input_ids
    return {
        "input_values": inputs.input_values[0],
        "labels": labels,
    }
dataset = dataset.map(
    prepare_batch,
    remove_columns=dataset["train"].column_names,
    desc="Preparing dataset"
)
# ==========================================================
# 8. DATA COLLATOR
# ==========================================================
def data_collator(features):
    input_values = [f["input_values"] for f in features]
    labels = [f["labels"] for f in features]
    batch = processor.pad(
        {"input_values": input_values},
        padding=True,
        return_tensors="pt"
    )
    with processor.as_target_processor():
        labels_batch = processor.pad(
            {"input_ids": labels},
            padding=True,
            return_tensors="pt"
        )
    batch["labels"] = labels_batch["input_ids"].masked_fill(
        labels_batch["input_ids"] == processor.tokenizer.pad_token_id,
        -100
    )
    return batch
# ==========================================================
# 9. METRICS
# ==========================================================
wer_metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred_str = processor.batch_decode(pred_ids)
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}
# ==========================================================
# 10. TRAINING
# ==========================================================
training_args = TrainingArguments(
    output_dir=VAD_OUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=1,
    learning_rate=3e-5,
    num_train_epochs=10,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    save_steps=500,
    eval_steps=500,
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    tokenizer=processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
# ==========================================================
# 11. RUN
# ==========================================================
if __name__ == "__main__":
    print("🚀 Start training")
    trainer.train()
    print("📊 Eval:")
    print(trainer.evaluate())
    trainer.save_model(VAD_OUT_DIR)
    processor.save_pretrained(VAD_OUT_DIR)
    print("✅ DONE")

Using cache found in C:\Users\DELL/.cache\torch\hub\snakers4_silero-vad_master


Train samples: 200
Eval samples : 100
🔪 VAD train
🔪 VAD eval


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Preparing dataset: 100%|██████████| 99/99 [00:00<00:00, 677.93 examples/s]
C:\Temp\ipykernel_11736\2446115365.py:209: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Start training


c:\ProgramData\miniconda3\envs\whisper-env\lib\site-packages\transformers\models\wav2vec2\processing_wav2vec2.py:180: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


Step,Training Loss
25,16.020000
50,4.114900
75,2.396100
100,1.637600
125,1.088500
150,0.947300
175,0.947700
200,0.825700
225,0.790500
250,0.810000


📊 Eval:


c:\ProgramData\miniconda3\envs\whisper-env\lib\site-packages\transformers\models\wav2vec2\processing_wav2vec2.py:180: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(


{'eval_loss': 0.7205044031143188, 'eval_wer': 0.9641975308641976, 'eval_runtime': 5.0291, 'eval_samples_per_second': 19.685, 'eval_steps_per_second': 19.685, 'epoch': 10.0}
✅ DONE


In [4]:
# ==========================================================
# 0. IMPORTS
# ==========================================================
import os
import torch
import torchaudio
import pandas as pd
import numpy as np
import soundfile as sf
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
# ==========================================================
# 1. PATHS & CONFIG
# ==========================================================
BASE_DIR = r"D:/tms/tms_study/voice to text/common-voice"
TRAIN_CSV = "cv-other-train.csv"
EVAL_CSV = "cv-valid-dev.csv"
TRAIN_AUDIO_DIR = os.path.join(BASE_DIR, "cv-other-train")
EVAL_AUDIO_DIR = os.path.join(BASE_DIR, "cv-valid-dev")
VAD_OUT_DIR = os.path.join(BASE_DIR, "whisper-common-voice-vad")
os.makedirs(VAD_OUT_DIR, exist_ok=True)
SEG_TRAIN_DIR = os.path.join(VAD_OUT_DIR, "train")
SEG_EVAL_DIR = os.path.join(VAD_OUT_DIR, "eval")
os.makedirs(SEG_TRAIN_DIR, exist_ok=True)
os.makedirs(SEG_EVAL_DIR, exist_ok=True)
MODEL_NAME = "openai/whisper-tiny"
SAMPLE_RATE = 16000
MAX_SEGMENT_SEC = 15.0
NUM_TRAIN_SAMPLES = 200
NUM_EVAL_SAMPLES = 100
# ==========================================================
# 2. LOAD SILERO VAD
# ==========================================================
vad_model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)
(get_speech_timestamps,
 save_audio,
 read_audio,
 VADIterator,
 collect_chunks) = utils
# ==========================================================
# 3. LOAD CSV
# ==========================================================
def load_csv(csv_name, audio_dir):
    # Пропускаем строки с ошибками
    df = pd.read_csv(os.path.join(BASE_DIR, csv_name), on_bad_lines="skip")
   
    # Переименовываем колонки в ожидаемые
    df = df.rename(columns={"filename": "path", "text": "sentence"})
   
    # Берем только нужные колонки
    df = df[["path", "sentence"]].dropna()
   
    # Полные пути к аудио
    df["audio_path"] = df["path"].apply(lambda x: os.path.join(audio_dir, x))
   
    # Оставляем только существующие файлы
    df = df[df["audio_path"].apply(os.path.exists)]
   
    return df.reset_index(drop=True)
train_df = load_csv(TRAIN_CSV, TRAIN_AUDIO_DIR)
eval_df = load_csv(EVAL_CSV, EVAL_AUDIO_DIR)
# Adjust samples if eval is empty
if len(eval_df) == 0:
    total_needed = NUM_TRAIN_SAMPLES + NUM_EVAL_SAMPLES
    train_df = train_df.head(total_needed)
    eval_df = train_df.tail(NUM_EVAL_SAMPLES).copy()
    train_df = train_df.head(NUM_TRAIN_SAMPLES)
else:
    train_df = train_df.head(NUM_TRAIN_SAMPLES)
    eval_df = eval_df.head(NUM_EVAL_SAMPLES)
print(f"Train samples: {len(train_df)}")
print(f"Eval samples : {len(eval_df)}")
# ==========================================================
# 4. VAD SEGMENTATION
# ==========================================================
def vad_segment(df, out_dir):
    rows = []
    for idx, row in df.iterrows():
        wav = read_audio(row["audio_path"], sampling_rate=SAMPLE_RATE)
        timestamps = get_speech_timestamps(
            wav,
            vad_model,
            sampling_rate=SAMPLE_RATE
        )
        if not timestamps:
            continue
        speech_chunks = []
        for ts in timestamps:
            dur = (ts["end"] - ts["start"]) / SAMPLE_RATE
            if dur > MAX_SEGMENT_SEC:
                continue
            speech_chunks.append(wav[ts["start"]:ts["end"]])
        if not speech_chunks:
            continue
        concat_speech = torch.cat(speech_chunks) if len(speech_chunks) > 1 else speech_chunks[0]
        total_dur = len(concat_speech) / SAMPLE_RATE
        if total_dur > MAX_SEGMENT_SEC:
            continue
        out_name = f"{idx}.wav"
        out_path = os.path.join(out_dir, out_name)
        sf.write(out_path, concat_speech.numpy(), SAMPLE_RATE)
        rows.append({
            "audio_path": out_path,
            "text": row["sentence"]
        })
    return pd.DataFrame(rows)
print("🔪 VAD train")
train_seg = vad_segment(train_df, SEG_TRAIN_DIR)
print("🔪 VAD eval")
eval_seg = vad_segment(eval_df, SEG_EVAL_DIR)
# ==========================================================
# 5. DATASET
# ==========================================================
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_seg),
    "eval": Dataset.from_pandas(eval_seg),
})
# ==========================================================
# 6. PROCESSOR + MODEL
# ==========================================================
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.forced_decoder_ids = None
# ==========================================================
# 7. PREPARE BATCH
# ==========================================================
def prepare_batch(batch):
    speech, sr = torchaudio.load(batch["audio_path"])
    speech = speech.squeeze()
    if sr != SAMPLE_RATE:
        speech = torchaudio.functional.resample(speech, sr, SAMPLE_RATE)
    speech = speech.numpy()
    inputs = processor(
        speech,
        sampling_rate=SAMPLE_RATE,
    )
    labels = processor.tokenizer(batch["text"]).input_ids
    return {
        "input_features": inputs.input_features[0],
        "labels": labels,
    }
dataset = dataset.map(
    prepare_batch,
    remove_columns=dataset["train"].column_names,
    desc="Preparing dataset"
)
# ==========================================================
# 8. DATA COLLATOR
# ==========================================================
class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
# ==========================================================
# 9. METRICS
# ==========================================================
wer_metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}
# ==========================================================
# 10. TRAINING
# ==========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir=VAD_OUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=1,
    learning_rate=3e-5,
    num_train_epochs=10,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    save_steps=500,
    eval_steps=500,
    report_to="none",
    predict_with_generate=True,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    tokenizer=processor.tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
# ==========================================================
# 11. RUN
# ==========================================================
if __name__ == "__main__":
    print("🚀 Start training")
    trainer.train()
    print("📊 Eval:")
    print(trainer.evaluate())
    trainer.save_model(VAD_OUT_DIR)
    processor.save_pretrained(VAD_OUT_DIR)
    print("✅ DONE")

Using cache found in C:\Users\DELL/.cache\torch\hub\snakers4_silero-vad_master


Train samples: 200
Eval samples : 100
🔪 VAD train
🔪 VAD eval


Preparing dataset: 100%|██████████| 99/99 [00:01<00:00, 67.08 examples/s]
C:\Temp\ipykernel_11736\1144058980.py:210: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


🚀 Start training


Step,Training Loss
25,3.380000
50,0.541200
75,0.323700
100,0.129700
125,0.059700
150,0.057700
175,0.007300
200,0.013100
225,0.005500
250,0.002100


c:\ProgramData\miniconda3\envs\whisper-env\lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


📊 Eval:


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


{'eval_loss': 0.6490723490715027, 'eval_wer': 0.23703703703703705, 'eval_runtime': 29.0418, 'eval_samples_per_second': 3.409, 'eval_steps_per_second': 3.409, 'epoch': 10.0}
✅ DONE


Этот словарь представляет собой итоговые метрики оценки после обучения модели распознавания речи Whisper на вашем датасете. Вот подробный разбор каждого ключа-значения, с сравнением с предыдущим результатом от Wav2Vec2 (где WER был ~96.4%, а loss ~0.72):

eval_loss: 0.6490723490715027
Средняя потеря (ошибка) на оценочном датасете. Это значение ниже, чем в Wav2Vec2 (~0.72), что указывает на меньшие общие ошибки в предсказаниях. Whisper справляется лучше, вероятно, благодаря своей архитектуре (seq2seq с трансформером) и предобучению на большом объёме многоязычных данных. Чем ниже loss, тем точнее модель в целом.
eval_wer: 0.23703703703703705
Word Error Rate (WER) — коэффициент ошибок слов, о котором мы говорили раньше. Здесь WER ~23.7%, что значительно лучше, чем ~96.4% в Wav2Vec2! Это значит, что модель правильно распознаёт около 76.3% слов (против всего 3.6% ранее). Улучшение огромно: Whisper изначально лучше адаптирован для разнообразной речи, включая акценты и шум. Однако 23.7% всё ещё не идеально для production (цель — ниже 10%), но с 200 сэмплами это хороший прогресс. Возможные причины: Whisper использует больше контекста и лучше справляется с сегментированными аудио после VAD.
eval_runtime: 29.0418
Время оценки на оценочном датасете, в секундах. Около 29 секунд — дольше, чем в Wav2Vec2 (~5 сек), из-за большего размера модели Whisper (tiny-версия, но всё равно сложнее) и, возможно, более длинных сэмплов. С маленьким датасетом (100 сэмплов) это нормально, но на большем объёме может вырасти.
eval_samples_per_second: 3.409
Скорость обработки: сколько аудиосэмплов модель оценивает в секунду. ~3.4 сэмпла/сек — медленнее, чем в Wav2Vec2 (~19.7), потому что Whisper требует больше вычислений (feature extraction и generation). Это типично для seq2seq-моделей.
eval_steps_per_second: 3.409
Шаги (батчи) обрабатываются в секунду во время оценки. Совпадает с samples/sec, так как размер батча — 1.
epoch: 10.0
Обучение завершилось после 10 эпох, как и в предыдущем случае.

В целом: Whisper показал гораздо лучшую точность (WER в 4 раза ниже!), подтверждая, что это более мощная модель для ASR-задач, особенно с многоязычными данными вроде Common Voice. Обучение завершилось успешно ("✅ DONE"). Для дальнейшего улучшения попробуйте больше сэмплов (например, 1000+), большую версию Whisper (small или base) или дообучение на конкретном языке. Если датасет на русском или другом неанглийском, Whisper multilingual справится лучше Wav2Vec2.

In [6]:
# ==========================================================
# 0. IMPORTS
# ==========================================================
import os
import torch
import torchaudio
import pandas as pd
import numpy as np
import soundfile as sf
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
# ==========================================================
# 1. PATHS & CONFIG
# ==========================================================
BASE_DIR = r"D:/tms/tms_study/voice to text/common-voice"
TRAIN_CSV = "cv-other-train.csv"
EVAL_CSV = "cv-valid-dev.csv"
TRAIN_AUDIO_DIR = os.path.join(BASE_DIR, "cv-other-train")
EVAL_AUDIO_DIR = os.path.join(BASE_DIR, "cv-valid-dev")
VAD_OUT_DIR = os.path.join(BASE_DIR, "whisper-common-voice-vad")
os.makedirs(VAD_OUT_DIR, exist_ok=True)
SEG_TRAIN_DIR = os.path.join(VAD_OUT_DIR, "train")
SEG_EVAL_DIR = os.path.join(VAD_OUT_DIR, "eval")
os.makedirs(SEG_TRAIN_DIR, exist_ok=True)
os.makedirs(SEG_EVAL_DIR, exist_ok=True)
MODEL_NAME = "openai/whisper-tiny"
SAMPLE_RATE = 16000
MAX_SEGMENT_SEC = 15.0
NUM_TRAIN_SAMPLES = 2000
NUM_EVAL_SAMPLES = 400
# ==========================================================
# 2. LOAD SILERO VAD
# ==========================================================
vad_model, utils = torch.hub.load(
    repo_or_dir="snakers4/silero-vad",
    model="silero_vad",
    trust_repo=True
)
(get_speech_timestamps,
 save_audio,
 read_audio,
 VADIterator,
 collect_chunks) = utils
# ==========================================================
# 3. LOAD CSV
# ==========================================================
def load_csv(csv_name, audio_dir):
    # Пропускаем строки с ошибками
    df = pd.read_csv(os.path.join(BASE_DIR, csv_name), on_bad_lines="skip")
   
    # Переименовываем колонки в ожидаемые
    df = df.rename(columns={"filename": "path", "text": "sentence"})
   
    # Берем только нужные колонки
    df = df[["path", "sentence"]].dropna()
   
    # Полные пути к аудио
    df["audio_path"] = df["path"].apply(lambda x: os.path.join(audio_dir, x))
   
    # Оставляем только существующие файлы
    df = df[df["audio_path"].apply(os.path.exists)]
   
    return df.reset_index(drop=True)
train_df = load_csv(TRAIN_CSV, TRAIN_AUDIO_DIR)
eval_df = load_csv(EVAL_CSV, EVAL_AUDIO_DIR)
# Adjust samples if eval is empty
if len(eval_df) == 0:
    total_needed = NUM_TRAIN_SAMPLES + NUM_EVAL_SAMPLES
    train_df = train_df.head(total_needed)
    eval_df = train_df.tail(NUM_EVAL_SAMPLES).copy()
    train_df = train_df.head(NUM_TRAIN_SAMPLES)
else:
    train_df = train_df.head(NUM_TRAIN_SAMPLES)
    eval_df = eval_df.head(NUM_EVAL_SAMPLES)
print(f"Train samples: {len(train_df)}")
print(f"Eval samples : {len(eval_df)}")
# ==========================================================
# 4. VAD SEGMENTATION
# ==========================================================
def vad_segment(df, out_dir):
    rows = []
    for idx, row in df.iterrows():
        wav = read_audio(row["audio_path"], sampling_rate=SAMPLE_RATE)
        timestamps = get_speech_timestamps(
            wav,
            vad_model,
            sampling_rate=SAMPLE_RATE
        )
        if not timestamps:
            continue
        speech_chunks = []
        for ts in timestamps:
            dur = (ts["end"] - ts["start"]) / SAMPLE_RATE
            if dur > MAX_SEGMENT_SEC:
                continue
            speech_chunks.append(wav[ts["start"]:ts["end"]])
        if not speech_chunks:
            continue
        concat_speech = torch.cat(speech_chunks) if len(speech_chunks) > 1 else speech_chunks[0]
        total_dur = len(concat_speech) / SAMPLE_RATE
        if total_dur > MAX_SEGMENT_SEC:
            continue
        out_name = f"{idx}.wav"
        out_path = os.path.join(out_dir, out_name)
        sf.write(out_path, concat_speech.numpy(), SAMPLE_RATE)
        rows.append({
            "audio_path": out_path,
            "text": row["sentence"]
        })
    return pd.DataFrame(rows)
print("🔪 VAD train")
train_seg = vad_segment(train_df, SEG_TRAIN_DIR)
print("🔪 VAD eval")
eval_seg = vad_segment(eval_df, SEG_EVAL_DIR)
# ==========================================================
# 5. DATASET
# ==========================================================
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_seg),
    "eval": Dataset.from_pandas(eval_seg),
})
# ==========================================================
# 6. PROCESSOR + MODEL
# ==========================================================
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.forced_decoder_ids = None
# ==========================================================
# 7. PREPARE BATCH
# ==========================================================
def prepare_batch(batch):
    speech, sr = torchaudio.load(batch["audio_path"])
    speech = speech.squeeze()
    if sr != SAMPLE_RATE:
        speech = torchaudio.functional.resample(speech, sr, SAMPLE_RATE)
    speech = speech.numpy()
    inputs = processor(
        speech,
        sampling_rate=SAMPLE_RATE,
    )
    labels = processor.tokenizer(batch["text"]).input_ids
    return {
        "input_features": inputs.input_features[0],
        "labels": labels,
    }
dataset = dataset.map(
    prepare_batch,
    remove_columns=dataset["train"].column_names,
    desc="Preparing dataset"
)
# ==========================================================
# 8. DATA COLLATOR
# ==========================================================
class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
# ==========================================================
# 9. METRICS
# ==========================================================
wer_metric = evaluate.load("wer")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": wer_metric.compute(predictions=pred_str, references=label_str)}
# ==========================================================
# 10. TRAINING
# ==========================================================
training_args = Seq2SeqTrainingArguments(
    output_dir=VAD_OUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=1,
    learning_rate=3e-5,
    num_train_epochs=10,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    save_steps=500,
    eval_steps=500,
    report_to="none",
    predict_with_generate=True,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    tokenizer=processor.tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
# ==========================================================
# 11. RUN
# ==========================================================
if __name__ == "__main__":
    print("🚀 Start training")
    trainer.train()
    print("📊 Eval:")
    print(trainer.evaluate())
    trainer.save_model(VAD_OUT_DIR)
    processor.save_pretrained(VAD_OUT_DIR)
    print("✅ DONE")

Using cache found in C:\Users\DELL/.cache\torch\hub\snakers4_silero-vad_master


Train samples: 2000
Eval samples : 400
🔪 VAD train
🔪 VAD eval


Preparing dataset: 100%|██████████| 391/391 [00:06<00:00, 63.75 examples/s]
C:\Temp\ipykernel_11736\861148920.py:210: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


🚀 Start training


You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
25,4.489800
50,2.262300
75,1.419100
100,0.874700
125,0.517000
150,0.586900
175,0.633500
200,0.612900
225,0.496900
250,0.503300


c:\ProgramData\miniconda3\envs\whisper-env\lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


📊 Eval:


{'eval_loss': 0.6033326983451843, 'eval_wer': 0.24519987409505822, 'eval_runtime': 114.0264, 'eval_samples_per_second': 3.429, 'eval_steps_per_second': 3.429, 'epoch': 10.0}
✅ DONE
